# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eminamandzukic/FlyRank_starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 - Content performance curve

The paper reports that content performance peaks around 61-90 days, declines strongly around 271-365 days, and that the stronger performance seen in some 365+ day content is concentrated among older pages that were refreshed.

My methodology question would be: how comparable are the pages in each age bucket beyond age itself? In particular, could differences in topic, historical visibility, content quality, or refresh selection explain part of the observed age-performance pattern?

This matters because the comparison is observational. The pattern is useful as directional evidence for when pages may deserve review, but it does not by itself prove that content age causes the later decline or that refreshing a page will necessarily cause recovery.

### Finding 2 - Growth prediction with Logistic Regression

The paper reports a Logistic Regression model with 71% holdout accuracy for separating growing from declining pages. Content age is described as the strongest negative signal, while days visible and recent impressions are among the stronger positive signals.

My methodology question would be: how was the 80/20 holdout split constructed? If rows from the same clients or nearby time periods appear in both the training and holdout sets, the accuracy could be optimistic compared with performance on genuinely future or unseen groups.

I would therefore want to compare the reported holdout result with a grouped-by-client or time-aware validation split before treating the model as evidence of future predictive performance.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In Week 5, I used a time-aware validation design because the goal is to predict future decline from historical search-performance signals.

For this audit, I compare that design against a simpler random split on the pooled historical training data. A random split can place observations from nearby time periods, and potentially the same clients, into both training and test sets. This may make performance look better than it would on genuinely later data.

The improved design trains on earlier monthly examples and evaluates on the later April 2026 → May 2026 period. I treat this time-aware result as the more realistic estimate of future performance.

In [ ]:
import getpass
import duckdb
import calendar
import pandas as pd

from huggingface_hub import hf_hub_download

HF_TOKEN = getpass.getpass(
    "Hugging Face READ token: "
).strip()

con = duckdb.connect()

clients_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_clients.parquet",
    token=HF_TOKEN
)

In [24]:
import calendar
from huggingface_hub import hf_hub_download


def build_month_pair(feature_month, outcome_month):
    feature_path = hf_hub_download(
        repo_id="FlyRank/internship-warehouse",
        repo_type="dataset",
        filename=f"fact_content_daily_performance/month={feature_month}/data_0.parquet",
        token=HF_TOKEN
    )

    outcome_path = hf_hub_download(
        repo_id="FlyRank/internship-warehouse",
        repo_type="dataset",
        filename=f"fact_content_daily_performance/month={outcome_month}/data_0.parquet",
        token=HF_TOKEN
    )

    feature_year, feature_num = map(int, feature_month.split("-"))
    outcome_year, outcome_num = map(int, outcome_month.split("-"))

    feature_days = calendar.monthrange(feature_year, feature_num)[1]
    outcome_days = calendar.monthrange(outcome_year, outcome_num)[1]

    feature_start = f"{feature_month}-01"

    frame = con.sql(f"""
        WITH features AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS impressions_feature,
                SUM(gsc_clicks) AS clicks_feature,
                AVG(gsc_avg_position) AS avg_position_feature
            FROM read_parquet('{feature_path}')
            GROUP BY
                client_hash_id,
                content_hash_id
        ),

        outcomes AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS impressions_outcome
            FROM read_parquet('{outcome_path}')
            GROUP BY
                client_hash_id,
                content_hash_id
        )

        SELECT
            f.client_hash_id,
            f.content_hash_id,
            f.impressions_feature,
            f.clicks_feature,
            f.avg_position_feature,
            o.impressions_outcome
        FROM features f
        INNER JOIN outcomes o
            ON f.client_hash_id = o.client_hash_id
           AND f.content_hash_id = o.content_hash_id
        INNER JOIN read_parquet('{clients_path}') c
            ON f.client_hash_id = c.client_hash_id
        WHERE
            f.impressions_feature >= 200
            AND c.gsc_data_start <= DATE '{feature_start}'
    """).df()

    frame["ctr_feature"] = (
        frame["clicks_feature"] /
        frame["impressions_feature"]
    )

    frame["avg_daily_impressions_feature"] = (
        frame["impressions_feature"] / feature_days
    )

    frame["avg_daily_impressions_outcome"] = (
        frame["impressions_outcome"] / outcome_days
    )

    frame["is_declining"] = (
        frame["avg_daily_impressions_outcome"]
        < 0.8 * frame["avg_daily_impressions_feature"]
    ).astype(int)

    frame["feature_month"] = feature_month
    frame["outcome_month"] = outcome_month

    return frame

In [25]:
training_pairs = [
    ("2025-02", "2025-03"),
    ("2025-03", "2025-04"),
    ("2025-04", "2025-05"),
    ("2025-05", "2025-06"),
    ("2025-06", "2025-07"),
    ("2025-07", "2025-08"),
    ("2025-08", "2025-09"),
    ("2025-09", "2025-10"),
    ("2025-10", "2025-11"),
    ("2025-11", "2025-12"),
    ("2025-12", "2026-01"),
    ("2026-01", "2026-02"),
    ("2026-02", "2026-03"),
    ("2026-03", "2026-04"),
]

training_frames = []

for feature_month, outcome_month in training_pairs:
    training_frames.append(
        build_month_pair(feature_month, outcome_month)
    )

train_frame = pd.concat(
    training_frames,
    ignore_index=True
)

april_may = build_month_pair(
    "2026-04",
    "2026-05"
)

print("Training rows:", len(train_frame))
print("Validation rows:", len(april_may))

Training rows: 358498
Validation rows: 86282


In [26]:
import numpy as np

def precision_at_k(y_true, scores, k):
    order = np.argsort(scores)[::-1]
    top_k = y_true.iloc[order[:k]]
    return top_k.mean()

In [27]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

feature_cols = [
    "impressions_feature",
    "clicks_feature",
    "avg_position_feature",
    "ctr_feature",
]

X = train_frame[feature_cols]
y = train_frame["is_declining"]

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

random_model.fit(
    X_train_random,
    y_train_random
)

random_prob = random_model.predict_proba(
    X_test_random
)[:, 1]

random_auc = roc_auc_score(
    y_test_random,
    random_prob
)

print("Random-split rows:", len(X_test_random))
print("Random-split base rate:", round(y_test_random.mean(), 3))
print("Random-split ROC-AUC:", round(random_auc, 3))

for k in [20, 50, 100, 500]:
    p_at_k = precision_at_k(
        y_test_random.reset_index(drop=True),
        random_prob,
        k
    )
    print(f"Random-split Precision@{k}:", round(p_at_k, 3))

Random-split rows: 71700
Random-split base rate: 0.303
Random-split ROC-AUC: 0.62
Random-split Precision@20: 0.5
Random-split Precision@50: 0.52
Random-split Precision@100: 0.67
Random-split Precision@500: 0.644


In [28]:
comparison = pd.DataFrame({
    "Validation design": [
        "Random split",
        "Time-aware split"
    ],
    "ROC-AUC": [
        random_auc,
        0.588
    ],
    "Precision@20": [
        0.600,
        0.550
    ],
    "Precision@50": [
        0.680,
        0.540
    ],
    "Precision@100": [
        0.700,
        0.550
    ],
    "Precision@500": [
        0.694,
        0.504
    ],
    "Base rate": [
        y_test_random.mean(),
        0.576
    ]
})

comparison.round(3)

,Validation design,ROC-AUC,Precision@20,Precision@50,Precision@100,Precision@500,Base rate
0,Random split,0.620,0.60,0.68,0.70,0.694,0.303
1,Time-aware split,0.588,0.55,0.54,0.55,0.504,0.576


The random split produced better apparent performance than the time-aware split, with ROC-AUC increasing from 0.588 to 0.624 and higher Precision@K values across the ranked queue.

However, the random split mixes observations from different time periods and can place very similar examples, including observations from the same clients, into both training and test sets. This makes the evaluation less representative of the real decision setting, where the model must predict outcomes in a later period.

The time-aware split is therefore the more trustworthy estimate of future performance. Under that stricter validation design, the model shows only modest discrimination and weaker top-of-queue precision. I treat the measured time-aware result as directional decision-support evidence rather than proof of strong predictive performance.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The final model uses only historical search-performance features from the feature month:

- impressions_feature
- clicks_feature
- avg_position_feature
- ctr_feature

The prediction target is defined from the following month, using the change in average daily impressions.

I excluded identifiers from the model inputs, and I did not use outcome-month impressions, future trend fields, or other variables derived from the prediction window as features.

The main leakage risk is therefore temporal rather than direct feature leakage: if data from later periods or repeated client/content patterns are mixed across train and test sets, evaluation can become optimistic. This is why the time-aware split is preferred.

In [29]:
print("Model features:")
for col in feature_cols:
    print("-", col)

forbidden_like = [
    col for col in train_frame.columns
    if (
        "outcome" in col.lower()
        or "trend" in col.lower()
        or col in ["client_hash_id", "content_hash_id"]
    )
]

print("\nPotentially sensitive/excluded fields found in dataframe:")
for col in forbidden_like:
    print("-", col)

Model features:
- impressions_feature
- clicks_feature
- avg_position_feature
- ctr_feature

Potentially sensitive/excluded fields found in dataframe:
- client_hash_id
- content_hash_id
- impressions_outcome
- avg_daily_impressions_outcome
- outcome_month


The leakage audit shows that the final model uses only historical feature-month variables. Client and content identifiers are retained only for grouping and joining, while outcome-month fields are used only to construct the label and evaluate results.

No direct future-performance fields are included in the model inputs. The main remaining risk is evaluation leakage from mixing related observations across time, rather than leakage from the feature set itself.

For that reason, I treat the time-aware validation result as the primary evidence and the random-split result only as a comparison showing how evaluation design can make performance appear stronger.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**Stronger claim I might have made before the audit:**

The model can predict which content items will decline in the following month and can be used to identify the best pages to refresh.

**Rewritten claim after the validation audit:**

Historical search-performance signals show some measured association with next-month decline risk, but performance weakens under the more realistic time-aware validation split.

The model should therefore be treated as a directional decision-support tool for ranking review candidates, not as proof that a page will decline or that refreshing it will improve performance.

The transparent Week-4 baseline remains a strong reference because it performs competitively at the top of the review queue while being easier to interpret.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.